# Lab 04: Data Visualization

## Overview
In this lab, we'll build upon our previous data exploration and enhance our understanding using data visualization.

Our goal is to gain deeper insights into our customer data, which is necessary for our ultimate objective: customer segmentation.



### Summary:

1. Load data and redo preprocessing from last lab 

2. Metric features visualizations
    - Univariate distribution 
    - Bivariate distribution 
    - Correlation Matrix 

7. Optional Exercice: project data visualization

# Setup

In [ ]:
# Remember: library imports are ALWAYS at the top of the script, no exceptions!
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

from itertools import product

## If you have an error here saying ydata_profiling is not available
## Run the instructions above
## IF you are still unable to install `ydata-profiling` through this method, 
## Comment out the line below importing ProfileReport
from ydata_profiling import ProfileReport

# for better resolution plots
%config InlineBackend.figure_format = 'retina' # optionally, you can change 'svg' to 'retina'

# Setting seaborn style
sns.set_theme()

# Context
The data we will be using through the pratical classes comes from a small relational database whose schema can be seen below: \
![alt text](../figures/schema.png "Relation database schema")

# Reading the Data

In [ ]:
# path to database
my_path = os.path.join("..", "data", "datamining.db")

# connect to the database
conn = sqlite3.connect(my_path)

# the query
query = """
select
    age, 
    income, 
    frq, 
    rcn, 
    mnt, 
    clothes, 
    kitchen, 
    small_appliances, 
    toys, 
    house_keeping,
    dependents, 
    per_net_purchase,
    g.gender, 
    e.education, 
    m.status, 
    r.description
from customers as c
    join genders as g on g.id = c.gender_id
    join education_levels as e on e.id = c.education_id
    join marital_status as m on m.id = c.marital_status_id
    join recommendations as r on r.id = c.recommendation_id
order by c.id;
"""

df = pd.read_sql_query(query, conn)

# Metadata
- *id* - The unique identifier of the customer
- *age* - The year of birht of the customer
- *income* - The income of the customer
- *frq* - Frequency: number of purchases made by the customer
- *rcn* - Recency: number of days since last customer purchase
- *mnt* - Monetary: amount of € spent by the customer in purchases
- *clothes* - Number of clothes items purchased by the customer
- *kitchen* - Number of kitchen items purchased by the customer
- *small_appliances* - Number of small_appliances items purchased by the customer
- *toys* - Number of toys items purchased by the customer
- *house_keeping* - Number of house_keeping items purchased by the customer
- *dependents* - Binary. Whether or not the customer has dependents
- *per_net_purchase* - Percentage of purchases made online
- *education* - Education level of the customer
- *status* - Marital status of the customer
- *gender* - Gender of the customer
- *description* - Last customer's recommendation description

# Initial Analysis

Pandas user guide: https://pandas.pydata.org/pandas-docs/stable/user_guide/index.html

Pandas 10 min tutorial: https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html

## Potential Problems:
- Duplicates?
- Data types?
- Missing values?
- Strange values?
- Descriptive statistics?

### Insights from previous notebook:

- Duplicates?
    - We didn't find any
- Data types?
    - `income` was being recognized as `object` (empty strings)
    - `dependents` was being recognized as `object` (empty strings)

<div align="center">

| Metric Features   | Non-Metric Features |
|-------------------|---------------------|
| age               | dependents          |
| income            | gender              |
| frq               | education           |
| rcn               | status              |
| mnt               | description         |
| clothes           |                     |
| kitchen           |                     |
| small_appliances  |                     |
| toys              |                     |
| house_keeping     |                     |

</div>


- Missing values?
    - replaced empty strings '""' with `np.nan` to reflect that these were missing values
- Strange values?
    - `education` has `OldSchool` category
- Descriptive statistics?
    - Metric: mean, median, min, ...
    - Non-metric: mode (top)


### Replicating modifications from previous notebook

In [ ]:
# replace "" by nans
df.replace("", np.nan, inplace=True)

# count of missing values
df.isna().sum()

In [ ]:
# fix wrong dtypes
df.dependents = df.dependents.astype("boolean")  # converting to "boolean" over "bool" allows preservation of NaNs

In [ ]:
# check dataset data types again
df.dtypes

In [ ]:
# check descriptive statistics again
df.describe().round(2).T

In [ ]:
# check descriptive statistics again
df.describe(include=["O", bool]).T

# Visual Exploration tools

Matplotlib tutorials: https://matplotlib.org/stable/tutorials/index.html

Matplotlib gallery: https://matplotlib.org/stable/gallery/index.html

Seaborn tutorials: https://seaborn.pydata.org/tutorial.html


Seaborn gallery: https://seaborn.pydata.org/examples/index.html

### Matplotlib vs Seaborn:

**Matplotlib** - lower level. allows to fully customize the plot appearance

**Seaborn** - higher level. Complex off-the-shelf plots with one line. Matplotlib on steroids


In [ ]:
#Define metric and non-metric features. Why?
non_metric_features = ["education", "status", "gender", "dependents", "description"]
metric_features = df.columns.drop(non_metric_features).to_list()

######## OR ########
# metric_features = ['age', 'income', 'frq', 'rcn', 'mnt', 
#                    'clothes', 'kitchen', 'small_appliances', 'toys', 'house_keeping', 
#                    'per_net_purchase']


## Pyplot-style vs Object-Oriented-style
- Explicitly create figures and axes, and call methods on them (the "object-oriented (OO) style").
- Rely on pyplot to automatically create and manage the figures and axes, and use pyplot functions for plotting.

More details: https://matplotlib.org/matplotblog/posts/pyplot-vs-object-oriented-interface/

# Numeric Variables' Univariate Distribution

Now that we have reviewed the data quality, let's begin visualizing the distribution of individual numerical features. Univariate analysis helps us understand the shape, central tendency, and spread of each variable, and can reveal outliers. We will use histograms to show the frequency distribution and boxplots to visualize the quartiles and potential outliers.

## Histograms

In [ ]:
# Using matplotlib
# Single Metric Variable Histogram
plt.hist(df["age"], bins=10)  # mess around with the bins
plt.title("age", y=-0.2)

plt.show()

In [ ]:
## Why change bins?

##########################################
## NO NEED to understand this code, this just generates some random numbers ##############
##########################################

rng = np.random.default_rng(84038410067234)

rn_size=500
rn_bins=8
rn_loc=1.75

hist_data = np.concatenate([rng.normal(loc=rn_loc, size=rn_size).round(5),
                            rng.normal(loc=-rn_loc, size=rn_size).round(5)])
##########################################


fig, axes = plt.subplots(2,1,figsize=(8,8), sharex=True, tight_layout=True)

sns.histplot(hist_data, 
             ax=axes[0],
             bins=5, ## This is the only thing we change
             )
sns.histplot(hist_data, 
             ax=axes[1],
             bins=20, ## This is the only thing we change
            )

axes[0].set_title('bins=5', ha='left', x=.8, y=.7, fontsize='x-large')
axes[1].set_title('bins=20', ha='left', x=.8, y=.7, fontsize='x-large')

plt.show()

### Exercise 1: Plot histogram for income, compare with the histogram of age

In [ ]:
# Single Metric Variable Histogram
#plt
plt.show()

In [ ]:
# Using seaborn
# Histogram for Age
sns.histplot(df['age'], bins=20, kde=True)
plt.title('Distribution of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

### Exercise 2: Plot histogram for mnt and frq in the same figure (HINT: Use subplots) 


In [ ]:
subplot_figsize = (8,4) # (width, height)

fig, axes = plt.subplots(
    1, # nrows
    2, # ncols
    figsize=subplot_figsize)

## Do something here

## 1. HINT: use axes[i].hist() instead of plt.hist()

## 2. HINT: don't forget to label each subplot!
axes[0].set_title('axes[0]') ## What should the title be?
axes[1].set_title('axes[1]') ## What should the title be?


plt.show()

Plotting ALL Numeric Variables' Histograms in one figure


In [ ]:
# Plot ALL Numeric Variables' Histograms in one figure

sp_rows = 2
sp_cols = 6


# Prepare figure. Create individual axes where each histogram will be placed
fig, axes = plt.subplots(sp_rows, 
                         sp_cols, 
                         figsize=(20, 11))

# Plot data
# Iterate across axes objects and associate each histogram (hint: use the ax.hist() instead of plt.hist()):
for ax, feat in zip(axes.flatten(), metric_features): # Notice the zip() function and flatten() method
    ax.hist(df[feat])
    ax.set_title(feat, y=-0.13)
    
# Layout
# Add a centered title to the figure:
title = "Numeric Variables' Histograms"

plt.suptitle(title)

if not os.path.exists(os.path.join('..', 'figures', 'eda')):
    # if the exp_analysis directory is not present then create it first
    os.makedirs(os.path.join('..', 'figures', 'eda'))
    
plt.savefig(os.path.join('..', 'figures', 'eda', 'numeric_variables_histograms.png'), dpi=200)

plt.show()

## Boxplots

Box plots elements are the first and third quartiles (Q1 and Q3), the median (Q2) and the whiskers, that stop on lowest and highest values that are not considered outliers. The formula to calculate the boundaries is: 

$$
  IQR = Q_3 - Q_1 \\\

  minimum = Q_1 - 1.5*IQR \\\

  maximum = Q_3 + 1.5*IQR 
$$

In [ ]:
# Single Metric Variable Box Plot
sns.boxplot(x=df["age"])

plt.show()

Plotting ALL Numeric Variables' Boxplots in one figure


In [ ]:
# Plot ALL Numeric Variables' Histograms in one figure

## What do these do?
sp_rows = 2
sp_cols = ceil(len(metric_features) / sp_rows)


# Prepare figure. Create individual axes where each histogram will be placed
fig, axes = plt.subplots(sp_rows, 
                         sp_cols, 
                         figsize=(20, 11),
                         tight_layout=True
                        )

# Plot data
# Iterate across axes objects and associate each histogram (hint: use the ax.hist() instead of plt.hist()):
for ax, feat in zip(axes.flatten(), metric_features): # Notice the zip() function and flatten() method
    sns.boxplot(x=df[feat], ax=ax)
    
# Layout
# Add a centered title to the figure:
title = "Numeric Variables' Box Plots"

plt.suptitle(title)

if not os.path.exists(os.path.join('..', 'figures', 'eda')):
    # if the eda directory is not present then create it first
    os.makedirs(os.path.join('..', 'figures', 'eda'))


plt.savefig(os.path.join('..', 'figures', 'eda', 'numeric_variables_boxplots.png'), dpi=200)


plt.show()

### Numerical univariate visualizations: 

- Histogram (bins)
- Boxplots

### Insights:
- univariate distributions
- potential univariate outliers

--------------------------------------

## How can we improve our visualizations?
Remember Lecture discussion

Remove chart-junk, increase data-ink

https://seaborn.pydata.org/tutorial/aesthetics.html

In [ ]:
## plot histogram for income, test different seaborn themes

##########################################
## NO NEED to understand this code
##########################################

seaborn_styles = ["darkgrid", "white", "ticks", "whitegrid"]

f = plt.figure(figsize=(12, 9))
gs = f.add_gridspec(2, 2)

for (style_i, sns_style) in enumerate(seaborn_styles):

    with sns.axes_style(sns_style): # sns.set_style(sns_style):
        ax = f.add_subplot(gs[style_i>=2, style_i%2])
        
        ax.hist(df["income"], bins=10)
        ax.set_title('sns.set_style("{}")'.format(sns_style))

f.tight_layout()


plt.show()

You can also optimize for the context in which you are using your visualizations:

In [ ]:
## plot histogram for income, test different seaborn themes

##########################################
## NO NEED to understand this code
##########################################

sns.set_theme() ## Reset to darkgrid


seaborn_ctx = ["paper", "notebook", "talk", "poster"]

f = plt.figure(figsize=(12, 9))
gs = f.add_gridspec(2, 2)

for (ctx_i, sns_ctx) in enumerate(seaborn_ctx):
    sns.set_context(sns_ctx) # this part 

    ax = f.add_subplot(gs[ctx_i>=2, ctx_i%2])

    ax.hist(df["income"], bins=10)
    ax.set_title('sns.set_context("{}")'.format(sns_ctx))

f.tight_layout()


plt.show()

In [ ]:
sns.set_theme() ## Reset to darkgrid

Using seaborn to set the style also changes the non-seaborn plot styles:

In [ ]:
## Try it here: 
sns.set_style(...) # code here

plt.hist(df["income"], bins=10)
plt.title("income", y=-0.2)

plt.show()

In [ ]:
## using seaborn histplot:
sns.set_style(...) # code here

sns.histplot(df["income"], bins=10)

plt.show()

### Exercise 3: Use a different seaborn style, then visualize boxplot for 'mnt'

In [ ]:
# Single Metric Variable Box Plot
# Use a different seaborn style, then visualize boxplot for 'mnt'


# Numerical Bivariate Analysis - Pairwise Relationship of Numerical Variables

Moving beyond individual variable distributions, we will now explore the relationships between pairs of numerical features.

Bivariate analysis helps us identify potential correlations and patterns that might not be evident when looking at variables in isolation.

We will use scatterplots and hexbin plots to visualize the relationship between two variables, and a correlation matrix to see the linear relationships among all numerical features.

### Scatter Plots

In [ ]:
# Pair of Metric Variables Scatter Plot
sns.set_style("whitegrid")

plt.scatter(df["age"], df["income"])
plt.xlabel("age")
plt.ylabel("income")

plt.show()

# Is this useful? 
# How can we improve this visualization?


In [ ]:
# Single Metric Variable Scatter plot

plt.scatter(df["age"], df["income"], 
            # Draw the edges of each point
            # edgecolors=""
           )


plt.xlabel("age")
plt.ylabel("income")

plt.show()

# Is this useful? 
# How can we improve this visualization?


In [ ]:
# Pair of Metric Variables Scatter Plot

plt.scatter(df["age"], df["income"], 
            # Draw the edges of each point
            # edgecolors="", 
            # Change opacity level of each point
            # alpha=1, 
           )


plt.xlabel("age")
plt.ylabel("income")

plt.show()

# Is this useful? 
# How can we improve this visualization?


Now for `age` and `mnt`

In [ ]:
# Pair of Metric Variables Scatter Plot

plt.scatter(...
            #color='none' # remove the fill color of each point
           )


plt.xlabel("age")
plt.ylabel("mnt")

plt.show()

# Is this useful? 
# How can we improve this visualization?


### Hexbin Plot

In [ ]:
# Pair of Metric Variables Hexbin Plot

sns.set_style('ticks')

fig, ax = plt.subplots()

hb = ax.hexbin(df["age"], df["income"], 
                gridsize=20)

ax.set_title("Hexagon binning for age and income")
cb = fig.colorbar(hb, ax=ax, label='counts')

plt.xlabel("age")
plt.ylabel("income")

plt.show()

# Is this useful? 
# How can we improve this visualization?


#### Exercise 4: Improve the previous hexbin. <br>Specify a custom color using the cmap parameter: `cmap=""`
**HINT** see default colormaps from matplotlib here: https://matplotlib.org/stable/users/explain/colors/colormaps.html

In [ ]:
## Improve the colors used here

# Pair of Metric Variables Hexbin Plot

sns.set_style('ticks')

fig, ax = plt.subplots()

hb = ax.hexbin(df["age"], df["income"], 
                gridsize=20,
               ## ADD cmap code HERE
              )

ax.set_title("Hexagon binning for age and income")
cb = fig.colorbar(hb, ax=ax, label='counts')

plt.show()

# Is this useful? 
# How can we improve this visualization?


### KDE plot
KDE plot for age and mnt

In [ ]:
sns.set_style('ticks')
fig, ax = plt.subplots()

sns.kdeplot(
    data=df,
    x="age",
    y="mnt",
)

plt.title("KDE plot for age and mnt")
plt.xlabel("age")
plt.ylabel("mnt")
plt.show()


### Jointplot

In [ ]:
# Using seaborn jointplot

sns.set_style(style="ticks")
sns.jointplot(data=df, x="age", y="mnt", kind="hex", )
plt.show()

#### Exercise 5: Create a jointplot with age on the x-axis, and frq on the y-axis <br>Specify a custom color using the color parameter: `color=`


**HINT** see default colormaps from matplotlib here: https://matplotlib.org/stable/users/explain/colors/colormaps.html

In [ ]:
## Exercise:
## Create a jointplot with age on the x-axis, and frq on the y-axis
## Specify a custom color using the color parameter: color=""

sns.set_theme(style="ticks")

### Pairplots: all the scatterplots in one figure

This figure will take some time to run

In [ ]:
# Setting pairplot
sns.pairplot(df[metric_features], diag_kind="hist")

# Layout
plt.subplots_adjust(top=0.95)
plt.suptitle("Pairwise Relationship of Numerical Variables", fontsize=20)

# if the eda directory is not present then create it first
if not os.path.exists(os.path.join('..', 'figures', 'eda')):
    os.makedirs(os.path.join('..', 'figures', 'eda'))
    
plt.savefig(os.path.join('..', 'figures', 'eda', 'pairwise_numeric_scatterplots.png'), dpi=200)
plt.show()

## Metric Variables' Correlation Matrix

In [ ]:
df[metric_features].corr()

In [ ]:
corr = df[metric_features].corr(method="pearson")

fig = plt.figure(figsize=(10, 8))

sns.heatmap(data=corr, annot=True, )

plt.show()

## How can we improve this figure?

#### Execise 6: Improve Correlation Matrix figure

##### 6.1 Round the values of the correlation matrix to 2 decimal places.
- Hint: use.round()

In [ ]:
# Obtain correlation matrix. 
corr = df[metric_features].corr(method="pearson")




##### 6.2 Show only VERY high or VERY low values <br>Build annotation matrix: only values above |0.5| will appear annotated in the plot

In [ ]:
## What does this look like?
mask_annot = np.absolute(corr.values) >= 0.5 


# Try to understand what this np.where() does
annot = np.where(mask_annot, corr.values, np.full(corr.shape,"")) 




In [ ]:
## Let's break this down:

# mask_annot
# corr.values
# np.full(corr.shape,"")

##### 6.3: Putting it all together and a few tweaks


6.3a. Adjust minimum, maximum, and center values of the color scale (HINT: use `vmin`, `vmax`, `center`)<br>
    - Why? HINT: What are the range of values you expect with correlations?

6.3b. Show cells as squares, add line borders for aesthetics (HINT: use `square` and `linewidths`)<br>

6.3c. Choose better colormap: Why? (HINT: use `cmap`)<br>


In [ ]:

# Prepare figure
fig = plt.figure(figsize=(12, 8))


# Plot heatmap of the correlation matrix
sns.heatmap(data=corr, 
            annot=annot, # Specify custom annotation
            fmt='s', # The annotation matrix now has strings, so we need to explicitly say this
            ########## 3a.
            ########## 3b.
            ########## 3b.
           )

## Once you are happy, save this as an image
# plt.savefig(os.path.join('..', 'figures', 'eda', 'correlation_matrix.png'), dpi=200)

plt.show()

## How can we improve this figure?

#### Homework

Show only the lower triangle of the correlation matrix. *Why?*

### A note on correlation coefficients

It is not enough to only look at the values of correlation between variables.

Remember `age` and `small_appliances`? 

In [ ]:
sns.set_style(style="white")
fig, ax = plt.subplots()
cat1 = 'age'
cat2 = 'small_appliances'

hb = ax.hexbin(df[cat1], df[cat2], 
               gridsize=20,
               cmap='Greys'
              )

ax.set_title("{} and {},\ncorr = {}".format(cat1, cat2, 
                                            np.round(df[[cat1,cat2]].corr().values[0,1],4)))
ax.set_xlabel(cat1)
ax.set_ylabel(cat2)

cb = fig.colorbar(hb, ax=ax, label='counts')

plt.show()



### Numericals Bivariate visualizations overview 
- Single Comparison (bivariate)
    - Scatter plot – raw point cloud
    - Hexbin plot – binning for dense data
    - KDE plot (2D density contour / filled) – smooth estimate of joint distribution
    - Joint plot 
        - Scatter / Hexbin / KDE 
        - Marginal Histogram 

- Combination of bivariate comparisons 
    - Pair plot  
        - Rid of scatter plots / KDE / hexbin for every numeric variable pair
        - Diagonal: histograms or KDE of single variables

- Correlation Matrix visualization 
    - Heatmap 

### Insights
- possible bivariate relationships
- potential bivariate outliers
- univariate distributions (diagonal)

# Optional Exercise

Download the [Amazing Creative Desk Coworking dataset](https://elearning.novaims.unl.pt/mod/folder/view.php?id=113931). Using the `acdc_dataset.csv` file, perform the same exercises that we did in this notebook.

Identify the metric and non-metric features in this dataset.

**Visualize the different variables present in this dataset. Are there any interesting relationships present?**

# During our Exploratory Data Analysis (EDA), we must also account for:
- Incoherencies
- Outliers
- Missing values
- Feature Engineering

# Questions?
